# Schema Validation

This notebook validates the cleaned KAUST and KFUPM datasets against a common schema, identifies records that fail validation rules, and separates valid and rejected records.

In [1]:
from pathlib import Path
import pandas as pd

project_root = Path("..")
interim_dir = project_root / "data" / "interim"

In [2]:
kaust_2023 = pd.read_csv(
    interim_dir / "KAUST_2023_cleaned.csv"
)

kaust_crossref = pd.read_csv(
    interim_dir / "KAUST_2024_2025_cleaned.csv"
)

kfupm = pd.read_csv(
    interim_dir / "KFUPM_cleaned.csv"
)

In [3]:
print("KAUST 2023:", kaust_2023.shape)
print("KAUST Crossref:", kaust_crossref.shape)
print("KFUPM:", kfupm.shape)

KAUST 2023: (928, 13)
KAUST Crossref: (124, 13)
KFUPM: (48, 13)


In [4]:
schema = pd.DataFrame([
    ["research_id", "string", False, "Non-empty, unique"],
    ["university", "string", False, "KAUST or KFUPM"],
    ["title", "string", False, "Non-empty"],
    ["authors", "string", False, "Non-empty"],
    ["publication_year", "integer", False, "2023-2026"],
    ["publication_date", "date", True, "Valid YYYY-MM-DD if available"],
    ["abstract", "string", True, "Text if available"],
    ["research_field", "string", True, "Text if available"],
    ["tech_category", "string", True, "Defined taxonomy when assigned"],
    ["journal", "string", True, "Text if available"],
    ["doi", "string", False, "Required, valid DOI format"],
    ["url", "string", False, "Must start with http:// or https://"],
    ["source", "string", False, "KAUST Repository, Crossref, or KFUPM EPrints"]
], columns=[
    "column",
    "expected_type",
    "nullable",
    "allowed_values_or_range"
])

schema

,column,expected_type,nullable,allowed_values_or_range
0,research_id,string,False,"Non-empty, unique"
1,university,string,False,KAUST or KFUPM
2,title,string,False,Non-empty
3,authors,string,False,Non-empty
4,publication_year,integer,False,2023-2026
5,publication_date,date,True,Valid YYYY-MM-DD if available
6,abstract,string,True,Text if available
7,research_field,string,True,Text if available
8,tech_category,string,True,Defined taxonomy when assigned
9,journal,string,True,Text if available


## KAUST Validation

In [5]:
# Combine the cleaned KAUST datasets before validation

kaust = pd.concat(
    [kaust_2023, kaust_crossref],
    ignore_index=True
)

expected_columns = schema["column"].tolist()

print("KAUST rows:", len(kaust))
print("KAUST columns:", len(kaust.columns))
print("Columns match schema:", kaust.columns.tolist() == expected_columns)
print("Duplicate research IDs:", kaust["research_id"].duplicated().sum())

KAUST rows: 1052
KAUST columns: 13
Columns match schema: True
Duplicate research IDs: 0


In [6]:
def validate_dataset(df, allowed_university, min_year, max_year):
    result = df.copy()

    errors = pd.Series(
        "",
        index=result.index,
        dtype="string"
    )

    required_columns = schema.loc[
        schema["nullable"] == False,
        "column"
    ].tolist()

    for column in required_columns:
        missing = (
            result[column].isna()
            | result[column].astype("string").str.strip().eq("")
        )

        errors.loc[missing] += f"{column} is missing; "

    invalid_university = (
        result["university"] != allowed_university
    )

    errors.loc[invalid_university] += "invalid university; "

    years = pd.to_numeric(
        result["publication_year"],
        errors="coerce"
    )

    invalid_year = (
        years.isna()
        | ~years.between(min_year, max_year)
    )

    errors.loc[invalid_year] += "invalid publication year; "

    invalid_url = ~result["url"].astype("string").str.match(
        r"^https?://",
        na=False
    )

    errors.loc[invalid_url] += "invalid URL; "

    doi_present = result["doi"].notna()

    invalid_doi = (
        doi_present
        & ~result["doi"].astype("string").str.match(
            r"^10\.\d{4,9}/\S+$",
            case=False,
            na=False
        )
    )

    errors.loc[invalid_doi] += "invalid DOI; "

    date_present = result["publication_date"].notna()

    # Same rule as the team validator: a populated date must be a real
    # calendar date written exactly as YYYY-MM-DD (no time or timezone).
    date_text = result["publication_date"].astype("string").str.strip()

    parsed_dates = pd.to_datetime(
        date_text,
        format="%Y-%m-%d",
        errors="coerce"
    )

    invalid_date = (
        date_present
        & (
            ~date_text.str.fullmatch(r"\d{4}-\d{2}-\d{2}", na=False)
            | parsed_dates.isna()
        )
    )

    errors.loc[invalid_date] += "invalid publication date; "

    duplicate_id = result["research_id"].duplicated(
        keep=False
    )

    errors.loc[duplicate_id] += "duplicate research_id; "

    result["validation_error"] = errors.str.rstrip("; ")

    validated = result[
        result["validation_error"] == ""
    ].copy()

    rejected = result[
        result["validation_error"] != ""
    ].copy()

    return validated, rejected

In [7]:
# Date rule tests (artificial records, not saved)
date_test = pd.DataFrame({
    "research_id": ["T1", "T2", "T3", "T4", "T5"],
    "university": "KAUST",
    "title": "Test",
    "authors": "Test Author",
    "publication_year": 2024,
    "publication_date": ["2024-02-29", "2024-02-30", "2024-02",
                         "2024-04-11 00:00:00+00:00", None],
    "abstract": None, "research_field": None, "tech_category": None,
    "journal": None,
    "doi": "10.1234/test",
    "url": "https://example.org",
    "source": "TEST"
})

ok, bad = validate_dataset(date_test, "KAUST", 2023, 2026)

assert set(ok["research_id"]) == {"T1", "T5"}
assert set(bad["research_id"]) == {"T2", "T3", "T4"}
assert bad["validation_error"].eq("invalid publication date").all()

print("Date validation tests passed.")

Date validation tests passed.


In [8]:
kaust_validated, kaust_rejected = validate_dataset(
    kaust,
    allowed_university="KAUST",
    min_year=2023,
    max_year=2025
)

print("KAUST total:", len(kaust))
print("KAUST validated:", len(kaust_validated))
print("KAUST rejected:", len(kaust_rejected))

kaust_rejected["validation_error"].value_counts()

KAUST total: 1052
KAUST validated: 938
KAUST rejected: 114


validation_error
doi is missing        113
authors is missing      1
Name: count, dtype: Int64

## KFUPM Validation

In [9]:
print("KFUPM rows:", len(kfupm))
print("KFUPM columns:", len(kfupm.columns))
print("Columns match schema:", kfupm.columns.tolist() == expected_columns)
print("Duplicate research IDs:", kfupm["research_id"].duplicated().sum())

kfupm_validated, kfupm_rejected = validate_dataset(
    kfupm,
    allowed_university="KFUPM",
    min_year=2023,
    max_year=2026
)

print("KFUPM validated:", len(kfupm_validated))
print("KFUPM rejected:", len(kfupm_rejected))

kfupm_rejected["validation_error"].value_counts()

KFUPM rows: 48
KFUPM columns: 13
Columns match schema: True
Duplicate research IDs: 0
KFUPM validated: 0
KFUPM rejected: 48


validation_error
doi is missing    48
Name: count, dtype: Int64

In [10]:
kaust_validated.to_csv(
    interim_dir / "KAUST_validated.csv",
    index=False
)

kaust_rejected.to_csv(
    interim_dir / "KAUST_rejected.csv",
    index=False
)

kfupm_validated.to_csv(
    interim_dir / "KFUPM_validated.csv",
    index=False
)

kfupm_rejected.to_csv(
    interim_dir / "KFUPM_rejected.csv",
    index=False
)

print("Validation files saved successfully.")

Validation files saved successfully.
